In [ ]:
import os
import json
import openai

# 1) OpenAI API 키 불러오기 (환경 변수)
openai.api_key = os.getenv("OPENAI_API_KEY")

'''
채용공고 json 파일을 불러와서 , 해당 파일을 기준으로 프롬프트 설정을해서 QA 세트를 생성한다
그리고, 생성된 QA 세트를 Gemma3 학습 포맷에 맞춰 input/target 형태로 변환하여 JSONL 파일로 저장한다.

채용공고 JSON 파일은 다음과 같은 구조를 가집니다:
{
    "제목": "채용공고 제목",
    "회사 소개": "회사에 대한 소개",
    "주요 업무": ["업무1", "업무2", ...],
    "우대 사항": ["우대사항1", "우대사항2", ...]
}

QA세트는 너무 일반적이지 않고, 채용공고의 특성을 반영하여 생성됩니다.

'''
def generate_qa_pairs_for_posting(posting: dict) -> list:
    """
    GPT-4o-mini로 QA 쌍을 생성하는 부분은 이전 예시와 동일합니다.
    반환값: [{"question": "...", "answer": "..."}, ...]
    """
    title        = posting.get("제목", "").strip()
    company_info = posting.get("회사 소개", "").strip()
    duties       = posting.get("주요 업무", [])
    preferences  = posting.get("우대 사항", []) or posting.get("우대사항", [])

    duties_text      = "\n".join(f"- {d.strip()}" for d in duties if d.strip())
    preferences_text = "\n".join(f"- {p.strip()}" for p in preferences if p.strip())

    prompt = f"""
아래는 하나의 채용공고 정보입니다.
'직무', '주요 업무', '우대사항', '회사 소개' 부분만 이용해서
질문(Question)과 답변(Answer) 쌍을 JSON 배열 형식으로 생성해 주세요.
채용 절차나 기타 불필요한 정보는 제외해 주세요.
출력 예시:
[
  {{
    "question": "질문 텍스트",
    "answer": "답변 텍스트"
  }},
  ...
]

---
채용공고 정보:
제목: {title}

회사 소개:
{company_info}

주요 업무:
{duties_text}

우대사항:
{preferences_text}

---

위 내용을 바탕으로 4~6개의 일관된 QA 쌍을 만들어 주세요.
JSON 배열 형태로, 공백이나 주석 없이 순수한 JSON만 반환해 주세요.
"""

    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "당신은 채용 공고를 바탕으로 QA 쌍을 생성하는 전문가입니다."},
            {"role": "user",   "content": prompt}
        ],
        temperature=0.0,
        max_tokens=800,
        top_p=1.0,
    )
    generated = response.choices[0].message.content.strip()

    try:
        qa_list = json.loads(generated)
    except json.JSONDecodeError:
        start = generated.find("[")
        end   = generated.rfind("]") + 1
        qa_list = json.loads(generated[start:end])

    return qa_list


def wrap_for_gemma3(question: str, answer: str) -> dict:
    """
    Gemma3 학습 포맷에 맞춰 input/target 을 생성해 줍니다.
    """
    # 1) input 문자열 생성
    input_text = (
        "<bos>\n"
        "<start_of_turn>user\n"
        "시스템: 너는 채용 공고 정보를 기반으로 QA를 수행하는 챗봇입니다.\n"
        "<end_of_turn>\n"
        "<start_of_turn>user\n"
        f"질문: {question}\n"
        "<end_of_turn>\n"
        "<start_of_turn>model"
    )

    # 2) target 문자열 생성
    target_text = f"{answer}\n<end_of_turn><eos>"

    return {"input": input_text, "target": target_text}


if __name__ == "__main__":
    # 3) 로컬 채용공고 JSON 불러오기 (리스트 형태)
    with open("job_postings.json", "r", encoding="utf-8") as f:
        job_postings = json.load(f)

    # 4) 전체 결과 저장용 리스트
    gemma3_data = []

    for posting in job_postings:
        title = posting.get("제목", "무제 공고")
        print(f"▶ QA 생성 중: {title}")
        qa_pairs = generate_qa_pairs_for_posting(posting)

        # 5) 각 QA 쌍마다 Gemma3 input/target 생성
        for qa in qa_pairs:
            q = qa.get("question", "").strip()
            a = qa.get("answer", "").strip()
            if not q or not a:
                continue
            wrapped = wrap_for_gemma3(q, a)
            gemma3_data.append(wrapped)

    # 6) JSONL 포맷으로 저장
    out_path = "job_postings_gemma3.jsonl"
    with open(out_path, "w", encoding="utf-8") as fout:
        for entry in gemma3_data:
            fout.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"\n✅ Gemma3 형식 JSONL 생성 완료 → {out_path}")
